# Task 12 — Diesel fuels: source download and benchmark preparation

## Goal

Download the SWRI/Eigenvector CSV archive, align the spectra with cetane number (`CN`), reproduce the fixed 80/20 split with `random_state=123`, and then remove rows whose `CN` is missing from each partition. Splitting before filtering is required to reproduce the published 298/83 partition.

The original download is held in a temporary directory and removed at the end. Only the generated benchmark CSV files remain in the task directory.

## Original source

- Dataset page: <https://eigenvector.com/resources/data-sets/>
- Archive: <https://eigenvector.com/wp-content/uploads/2019/06/SWRI_Diesel_NIR_CSV.zip>

## Setup

Set the task parameters, locate the output directory, and define download and verification helpers. The comparison runs before the generated files replace any existing local copies.

In [1]:
TASK_FOLDER = "12-NIR_DieselFuels_cn"
INCLUDE_INDEX = True
COMPARISON_ATOL = 1e-12
SOURCE_URL = "https://eigenvector.com/wp-content/uploads/2019/06/SWRI_Diesel_NIR_CSV.zip"

In [2]:
from pathlib import Path
from urllib.request import Request, urlopen
import hashlib
import io
import shutil
import tempfile
import zipfile

import numpy as np
import pandas as pd
from scipy.io import loadmat


def find_task_dir(folder_name):
    """Support running from either the task directory or repository root."""
    candidates = [Path.cwd(), Path.cwd() / "datasets" / folder_name]
    for candidate in candidates:
        if candidate.name == folder_name and (candidate / "download_data.md").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        f"Run this notebook from {folder_name!r} or from the repository root."
    )


def download(url, destination):
    """Download a source file without retaining it in the repository."""
    request = Request(url, headers={"User-Agent": "NIRBENCH-DL/0.1 dataset preparation"})
    with urlopen(request, timeout=120) as response, destination.open("wb") as output:
        shutil.copyfileobj(response, output)
    print(f"Downloaded {destination.name} ({destination.stat().st_size:,} bytes)")


def csv_roundtrip(frame, include_index):
    """Represent a generated frame as it will appear after CSV serialization."""
    buffer = io.StringIO()
    frame.to_csv(buffer, index=include_index, lineterminator="\r\n")
    buffer.seek(0)
    return pd.read_csv(buffer)


def verify_and_write(train_frame, test_frame, include_index, atol):
    """Compare with published partitions when present, then write both CSV files."""
    results = []
    for split, generated in (("train", train_frame), ("test", test_frame)):
        output_path = TASK_DIR / f"data_{split}.csv"
        serialized = csv_roundtrip(generated, include_index)

        if output_path.exists():
            published = pd.read_csv(output_path)
            pd.testing.assert_frame_equal(
                serialized,
                published,
                check_dtype=False,
                check_exact=False,
                rtol=0,
                atol=atol,
            )
            max_delta = np.max(
                np.abs(
                    serialized.select_dtypes(include="number").to_numpy(dtype=float)
                    - published.select_dtypes(include="number").to_numpy(dtype=float)
                )
            )
            print(
                f"{split}: matches the published partition "
                f"(shape={serialized.shape}, maximum absolute difference={max_delta:.3g})"
            )
        else:
            print(f"{split}: no published CSV was present; generated shape={serialized.shape}")

        generated.to_csv(
            output_path,
            index=include_index,
            lineterminator="\r\n",
        )
        digest = hashlib.sha256(output_path.read_bytes()).hexdigest()
        results.append({
            "partition": split,
            "rows": len(generated),
            "spectral_variables": generated.shape[1] - 1,
            "sha256": digest,
        })

    return pd.DataFrame(results)


TASK_DIR = find_task_dir(TASK_FOLDER)
temporary_download = tempfile.TemporaryDirectory(prefix=f"nirbench_{TASK_FOLDER}_")
SOURCE_DIR = Path(temporary_download.name)
print(f"Preparing task {TASK_FOLDER}")

Preparing task 12-NIR_DieselFuels_cn


## Download and prepare

Download the authoritative source and apply the task specific conversion recorded in `DATA_LICENSES.md`.

In [3]:
from sklearn.model_selection import train_test_split

archive_path = SOURCE_DIR / "SWRI_Diesel_NIR_CSV.zip"
download(SOURCE_URL, archive_path)
with zipfile.ZipFile(archive_path) as archive:
    archive.extractall(SOURCE_DIR)

spectra_path = next(SOURCE_DIR.rglob("diesel_spec.csv"))
properties_path = next(SOURCE_DIR.rglob("diesel_prop.csv"))

# The Eigenvector CSV files contain metadata before their tabular headers.
spectra = pd.read_csv(spectra_path, skiprows=9)
properties = pd.read_csv(properties_path, skiprows=8)
assert len(spectra) == len(properties) == 784
assert spectra.iloc[:, 1].astype(str).equals(properties.iloc[:, 1].astype(str))

all_data = spectra.iloc[:, 2:-1].copy()  # 750–1550 nm; omit empty and label columns
all_data = all_data.assign(CN=properties["CN"])

# Preserve the historical order of operations: split all rows, then remove missing CN.
train_data, test_data = train_test_split(
    all_data,
    test_size=0.20,
    random_state=123,
)
train_data = train_data.dropna(subset=["CN"])
test_data = test_data.dropna(subset=["CN"])

assert train_data.shape == (298, 402)
assert test_data.shape == (83, 402)
print("Applied split-before-filter preparation and retained 401 spectral variables.")

Downloaded SWRI_Diesel_NIR_CSV.zip (1,029,110 bytes)
Applied split-before-filter preparation and retained 401 spectral variables.


## Verify and write

When published CSVs are present, compare their shapes, columns, row order, indices, and numerical values with the generated partitions. Then write `data_train.csv` and `data_test.csv` using deterministic CRLF line endings.

In [4]:
summary = verify_and_write(
    train_data,
    test_data,
    include_index=INCLUDE_INDEX,
    atol=COMPARISON_ATOL,
)
display(summary)
temporary_download.cleanup()
print("Temporary source files removed. Preparation complete.")

train: matches the published partition (shape=(298, 403), maximum absolute difference=1.04e-16)
test: matches the published partition (shape=(83, 403), maximum absolute difference=1.01e-16)


,partition,rows,spectral_variables,sha256
0,train,298,401,182e00a03b17a548c78f5f45ffd9825a1d66cadedd1be7...
1,test,83,401,b15b4bbec82b8ec91c0ce301275cc5cfa6d46f689a3973...


Temporary source files removed. Preparation complete.
